# SWE-bench：在隔离仓库中验证代码 Agent 补丁

**面试问题：一个代码 Agent 的补丁为什么必须经过沙箱、可见测试和隐藏边界测试？**

## 回答主线

1. SWE-bench 类任务评估的是补丁应用后仓库行为，不是模型能否描述一个可能的修复。
2. 执行器必须从干净快照创建隔离工作区，只允许补丁触达授权文件。
3. 静态门禁先拒绝危险导入和越界路径，再在子进程中运行确定性测试。
4. 可见测试通过只说明拟合了已知样本，不能排除把常量或狭窄条件写死。
5. 隐藏边界、回归测试和补丁最小性共同防止过拟合修复。
6. 生产沙箱还需要容器、资源上限、网络隔离、系统调用限制和完整审计。

## 真实案例

一个会员折扣仓库把满 100 元写成了大于 100 元，导致边界订单不打折。五个可见测试没有覆盖高年限会员，两个候选补丁都能修正 100 元边界，但其中一个把会员年限写死为 2，会在隐藏样本失败；另有一个包含 os 导入的危险候选被静态门禁拒绝。教学实验使用可读的小数据解释机制，结果不能外推为线上收益。

### 输入预览：Issue、源码与五个可见样本

In [1]:
import ast  # 导入语法树以检查候选补丁的危险能力。
import shutil  # 导入目录清理工具以在最后释放教学沙箱。
import subprocess  # 导入子进程以隔离执行候选源码。
import sys  # 导入当前 Python 解释器路径以保证环境一致。
import tempfile  # 导入临时目录以创建一次性仓库快照。
from pathlib import Path  # 导入路径对象以管理沙箱文件。

issue = "订单总额满100元且会员满2年应打9折，但100元边界未生效"  # 定义可复现的代码问题。
buggy_source = "def loyalty_discount(total, years):\n    if total > 100 and years >= 2:\n        return round(total * 0.9, 2)\n    return total\n"  # 构造含严格大于错误的仓库源码。
visible_cases = [(99, 2, 99), (100, 2, 90.0), (101, 2, 90.9), (100, 1, 100), (0, 5, 0)]  # 定义五个具有业务含义的可见测试样本。
sandbox_root = Path(tempfile.mkdtemp(prefix="swe_teaching_"))  # 创建与真实工作区隔离的临时仓库。
(sandbox_root / "discount.py").write_text(buggy_source, encoding="utf-8")  # 把问题源码写入沙箱而不触碰项目文件。
print("Issue：", issue)  # 输出 Agent 收到的问题描述。
print("问题源码：\n" + buggy_source)  # 展示需要修改的具体代码。
print("可见测试：", visible_cases)  # 展示输入、会员年限和期望折扣结果。

Issue： 订单总额满100元且会员满2年应打9折，但100元边界未生效
问题源码：
def loyalty_discount(total, years):
    if total > 100 and years >= 2:
        return round(total * 0.9, 2)
    return total

可见测试： [(99, 2, 99), (100, 2, 90.0), (101, 2, 90.9), (100, 1, 100), (0, 5, 0)]


## Baseline 基线：原始仓库运行可见测试

In [2]:
def write_test_runner(root, cases, filename):  # 把结构化样本写成独立 Python 测试脚本。
    literal_cases = repr(cases)  # 将受控测试数据序列化为 Python 字面量。
    runner = "from discount import loyalty_discount\nCASES = " + literal_cases + "\nfailures = []\nfor total, years, expected in CASES:\n    actual = loyalty_discount(total, years)\n    if actual != expected:\n        failures.append((total, years, expected, actual))\nprint('CASES', len(CASES))\nprint('FAILURES', failures)\nraise SystemExit(1 if failures else 0)\n"  # 构造只依赖目标函数的测试程序。
    (root / filename).write_text(runner, encoding="utf-8")  # 保存测试脚本到隔离仓库。

def run_test_file(root, filename):  # 在独立解释器中执行一个测试文件。
    bootstrap = "import runpy, sys\nsys.path.insert(0, '.')\nrunpy.run_path(sys.argv[1], run_name='__main__')"  # 在隔离模式中只显式加入当前候选目录并运行测试文件。
    completed = subprocess.run([sys.executable, "-I", "-c", bootstrap, filename], cwd=root, capture_output=True, text=True, timeout=10)  # 禁用用户 site 并限制执行时间。
    lines = [line for line in completed.stdout.splitlines() if line]  # 提取稳定输出而忽略运行耗时。
    return completed.returncode, lines  # 返回退出码和可读失败列表。

write_test_runner(sandbox_root, visible_cases, "visible_tests.py")  # 写入五个可见测试。
baseline_code, baseline_output = run_test_file(sandbox_root, "visible_tests.py")  # 在子进程运行原始仓库。
print(f"Baseline returncode={baseline_code}")  # 展示测试套件失败状态。
for line in baseline_output:  # 逐行展示真实失败样本。
    print(line)  # 显示 100 元边界的 expected 与 actual。

Baseline returncode=1
CASES 5
FAILURES [(100, 2, 90.0, 100)]


### 核心实现：补丁静态门禁与隔离候选工作区

In [3]:
overfit_source = "def loyalty_discount(total, years):\n    if total >= 100 and years == 2:\n        return round(total * 0.9, 2)\n    return total\n"  # 构造能过可见测试但把年限写死的过拟合补丁。
good_source = "def loyalty_discount(total, years):\n    if total >= 100 and years >= 2:\n        return round(total * 0.9, 2)\n    return total\n"  # 构造只修复比较符且保留原业务规则的最小补丁。
dangerous_source = "import os\ndef loyalty_discount(total, years):\n    os.system('echo unsafe')\n    return total\n"  # 构造包含系统调用能力的危险候选。

def static_gate(source):  # 用 AST 实现最小危险能力门禁。
    tree = ast.parse(source)  # 解析候选代码而不执行它。
    denied_imports = {"os", "subprocess", "socket"}  # 定义教学沙箱拒绝的高风险模块。
    reasons = []  # 收集所有拒绝原因。
    for node in ast.walk(tree):  # 遍历候选语法树。
        if isinstance(node, ast.Import):  # 检查普通 import 语句。
            for alias in node.names:  # 检查该语句中的每个模块。
                if alias.name.split(".")[0] in denied_imports:  # 判断顶层模块是否在拒绝集合。
                    reasons.append(f"denied-import:{alias.name}")  # 保存具体危险模块。
        if isinstance(node, ast.ImportFrom) and (node.module or "").split(".")[0] in denied_imports:  # 检查 from import 语句。
            reasons.append(f"denied-import:{node.module}")  # 保存具体危险模块。
    return len(reasons) == 0, reasons  # 返回门禁结论和可审计原因。

def evaluate_candidate(name, source, cases, test_name):  # 在独立候选目录中应用源码并运行指定测试。
    candidate_root = sandbox_root / name  # 为每个候选分配独立工作区。
    candidate_root.mkdir(exist_ok=True)  # 创建候选目录。
    allowed, reasons = static_gate(source)  # 执行静态能力门禁。
    if not allowed:  # 危险候选不允许进入动态执行阶段。
        return {"name": name, "allowed": False, "returncode": None, "output": reasons, "root": candidate_root}  # 返回拒绝证据。
    (candidate_root / "discount.py").write_text(source, encoding="utf-8")  # 只写授权目标文件。
    write_test_runner(candidate_root, cases, test_name)  # 为当前候选创建隔离测试。
    returncode, output = run_test_file(candidate_root, test_name)  # 在子进程执行候选实现。
    return {"name": name, "allowed": True, "returncode": returncode, "output": output, "root": candidate_root}  # 返回门禁和测试结果。

candidates = [("overfit", overfit_source), ("minimal", good_source), ("dangerous", dangerous_source)]  # 汇总三个候选补丁。
visible_results = [evaluate_candidate(name, source, visible_cases, "visible_tests.py") for name, source in candidates]  # 对所有候选运行可见测试。
print("候选      静态允许  可见测试码  输出")  # 输出候选评估表头。
for result in visible_results:  # 逐候选展示门禁和测试结果。
    print(f"{result['name']:<10} {str(result['allowed']):<8} {str(result['returncode']):<10} {result['output']}")  # 展示危险候选未被执行。

候选      静态允许  可见测试码  输出
overfit    True     0          ['CASES 5', 'FAILURES []']
minimal    True     0          ['CASES 5', 'FAILURES []']
dangerous  False    None       ['denied-import:os']


## 结果解读：可见测试不足以区分过拟合补丁

In [4]:
visible_passed = [result["name"] for result in visible_results if result["allowed"] and result["returncode"] == 0]  # 找出通过全部可见测试的候选。
dangerous_result = next(result for result in visible_results if result["name"] == "dangerous")  # 读取危险候选的门禁结果。
print("通过可见测试的候选：", visible_passed)  # 展示过拟合和最小补丁都暂时通过。
print("危险候选证据：", dangerous_result["output"])  # 展示静态门禁的具体拒绝原因。
print("补丁差异语义：overfit 把 years>=2 收窄为 years==2；minimal 只把 total>100 修为 total>=100。")  # 解释代码审查为何仍必要。
print("解读：执行可见测试证明修了已知样本，AST 门禁证明未授予明显危险能力，但二者都还不能证明补丁泛化。")  # 明确当前证据边界。

通过可见测试的候选： ['overfit', 'minimal']
危险候选证据： ['denied-import:os']
补丁差异语义：overfit 把 years>=2 收窄为 years==2；minimal 只把 total>100 修为 total>=100。
解读：执行可见测试证明修了已知样本，AST 门禁证明未授予明显危险能力，但二者都还不能证明补丁泛化。


## 失败案例：写死 years==2 通过可见测试却败在隐藏边界

In [5]:
hidden_cases = [(100, 5, 90.0), (250, 8, 225.0), (100, 2, 90.0), (99, 10, 99), (-1, 3, -1)]  # 构造高年限、金额边界和异常金额五个隐藏样本。
hidden_rows = []  # 收集两个安全候选的隐藏测试结果。
for name, source in candidates[:2]:  # 只对通过静态门禁的两个候选运行隐藏测试。
    result = evaluate_candidate(name + "_hidden", source, hidden_cases, "hidden_tests.py")  # 在新的干净工作区运行隐藏样本。
    hidden_rows.append(result)  # 保存当前候选结果。
print("候选      隐藏测试码  失败详情")  # 输出隐藏评估表头。
for result in hidden_rows:  # 逐候选展示泛化结果。
    print(f"{result['name']:<14} {result['returncode']:<10} {result['output']}")  # 展示过拟合补丁在 5 年和 8 年会员上失败。
print("修正策略：从 Issue 推导边界等价类，加入高年限与金额边界隐藏测试，并偏好语义最小的单运算符修改。")  # 总结防过拟合方法。

候选      隐藏测试码  失败详情
overfit_hidden 1          ['CASES 5', 'FAILURES [(100, 5, 90.0, 100), (250, 8, 225.0, 250)]']
minimal_hidden 0          ['CASES 5', 'FAILURES []']
修正策略：从 Issue 推导边界等价类，加入高年限与金额边界隐藏测试，并偏好语义最小的单运算符修改。


### 生产边界与补丁证据包

In [6]:
evidence_bundle = {"issue": issue, "allowed_file": "discount.py", "visible_cases": len(visible_cases), "hidden_cases": len(hidden_cases), "static_denials": dangerous_result["output"], "selected_patch": "minimal"}  # 构造补丁验收证据包。
print("补丁证据包：", evidence_bundle)  # 展示决定接受补丁所依据的完整信息。
print("生产替换点：真实 SWE-bench 沙箱还需只读基础镜像、网络禁用、CPU/内存/时间限制、补丁路径白名单、依赖缓存和仓库级回归测试。")  # 明确本机临时目录并非安全容器。

补丁证据包： {'issue': '订单总额满100元且会员满2年应打9折，但100元边界未生效', 'allowed_file': 'discount.py', 'visible_cases': 5, 'hidden_cases': 5, 'static_denials': ['denied-import:os'], 'selected_patch': 'minimal'}
生产替换点：真实 SWE-bench 沙箱还需只读基础镜像、网络禁用、CPU/内存/时间限制、补丁路径白名单、依赖缓存和仓库级回归测试。


## 回归测试：最后只保护缺陷复现、门禁和隐藏泛化

In [7]:
overfit_hidden = next(result for result in hidden_rows if result["name"] == "overfit_hidden")  # 读取过拟合补丁的隐藏结果。
minimal_hidden = next(result for result in hidden_rows if result["name"] == "minimal_hidden")  # 读取最小补丁的隐藏结果。
assert baseline_code != 0  # 验证原始仓库稳定复现 100 元边界缺陷。
assert set(visible_passed) == {"overfit", "minimal"}  # 验证可见测试无法区分两个候选。
assert not dangerous_result["allowed"] and dangerous_result["returncode"] is None  # 验证危险候选在执行前被拒绝。
assert overfit_hidden["returncode"] != 0 and minimal_hidden["returncode"] == 0  # 验证隐藏测试淘汰写死年限的补丁。
assert (minimal_hidden["root"] / "discount.py").read_text(encoding="utf-8") == good_source  # 验证最终工作区应用的是预期最小源码。
shutil.rmtree(sandbox_root)  # 在完成所有验证后清理一次性教学沙箱。
print("回归测试通过：缺陷复现、隔离执行、危险门禁、可见过拟合和隐藏泛化均成立，临时目录已清理。")  # 用少量断言总结代码 Agent 合同。

回归测试通过：缺陷复现、隔离执行、危险门禁、可见过拟合和隐藏泛化均成立，临时目录已清理。
